# 4 - GEODE: Building the Oracle

GEODE turns a document into a trained, self-corrected store: ingest, a propose -> diagnose -> repair loop that drops geometrically-implausible triples, then training and calibration. In this notebook we **run** a real (small) build, inspect what it produced, then look at the full production store.

## Run GEODE on a small document
A well-formed markdown table is enough. `build_rag_store` runs the `GeodeLoop` self-correction orchestrator and a geometry-supervised embedding loop, then trains the production store (small config here so it runs in seconds on CPU).

In [1]:
import tempfile, torch
from pathlib import Path
from knowlytix.knowledge.geode import build_rag_store, make_default_trainer
from knowlytix.knowledge.config import DocGMSConfig

doc = Path(tempfile.mktemp(suffix='.md'))
doc.write_text('''# Bank Fee Schedule\n\n## Fee Schedule\n\n| product | fee_amount | type |\n| --- | --- | --- |\n| overdraft | 35.00 | per_occurrence |\n| wire_international | 45.00 | per_transaction |\n| stop_payment | 30.00 | per_request |\n''')

cfg = DocGMSConfig(store_path=tempfile.mkdtemp(), ingest_mode='regex')
cfg.train.epochs = 60                       # small, for a fast notebook build
res = build_rag_store(doc, cfg, device=torch.device('cpu'),
                      geode_trainer=make_default_trainer(torch.device('cpu'), epochs=60),
                      max_iters=3)
print(f'converged={res.converged} iters={res.iterations} '
      f'entities={res.n_entities} triples={res.n_triples} enm={res.n_enm} '
      f'corrections={len(res.corrections)}')

knowlytix-core v0.2.0 licensed to customer=KnowlytixAgentBuilder tier=enterprise expires=2027-06-04


  GMS entities:  10
  GMS relations: 3
  GMS triples:   9
  Training data prepared:
    Triples:              9
    Agreement pairs:      3
    Contradiction pairs:  0
    Relation triangles:   0

Training GMS (cap): 10 entities, 3 relations, 9 triples
  Config: d_v=64, m=32, epochs=60, batch=64, neg=16+8 boundary
  Cap:    rho_max=1.5708, alpha=4.0, margin=0.3, use_diag=True
  Weights: pos=5.0, neg=2.0, shrink=0.05, barrier=0.01, lambda_tension=0.5
  Device: cpu
  Epoch    1/60  Loss: 26.4318  (L_pos=4.5206, L_neg=1.6614, rho_mean=0.392, rho_max=0.392, L_te=0.9765)
  Epoch   20/60  Loss: 3.2730  (L_pos=0.4019, L_neg=0.5149, rho_mean=0.419, rho_max=0.420, L_te=0.3601)
  Epoch   40/60  Loss: 2.0475  (L_pos=0.1800, L_neg=0.4409, rho_mean=0.432, rho_max=0.441, L_te=0.3413)
  Epoch   60/60  Loss: 1.4475  (L_pos=0.1497, L_neg=0.2199, rho_mean=0.437, rho_max=0.459, L_te=0.2915)
  GMS entities:  9
  GMS relations: 2
  GMS triples:   6


/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


  Training data prepared:
    Triples:              6
    Agreement pairs:      0
    Contradiction pairs:  0
    Relation triangles:   0

Training GMS: 9 entities, 2 relations, 6 triples
  Config: d_v=256, m=128, epochs=60, batch=256, neg=32
  Loss:   lambda_geo=5.0, lambda_tension=0.0, lambda_neg=0.5, gamma=1.0, adv_temp=2.0
  Device: cpu
  Epoch    1/60  Loss: 8.0887  (L_pos=1.3935, L_neg=0.4484)
  Epoch   20/60  Loss: 7.0382  (L_pos=1.2317, L_neg=0.3519)
  Epoch   40/60  Loss: 6.2133  (L_pos=1.0666, L_neg=0.3521)
  Epoch   60/60  Loss: 5.0371  (L_pos=0.9177, L_neg=0.1794)
  ENM populated: 3 entries
  Store saved to /tmp/tmploj9ua56
converged=True iters=1 entities=9 triples=6 enm=3 corrections=0


The build returns a `GMSExpertStore` trained on the surviving triples. We query it directly --- the same primitives as Chapter 3, now on a store we just built.

In [ ]:
print('asserted edges:', res.store.query_triples(head='overdraft'))
print('score_triple(overdraft, has_fee_amount, 35.0) =',
      round(res.store.score_triple('overdraft', 'has_fee_amount', '35.0'), 3))

## What the loop does
The `GeodeLoop` proposes the regex-extracted triples, diagnoses the ones that sit implausibly far on the manifold (or contradict an established fact) and repairs or drops them, recording every correction. On a clean table there is little to correct; on noisy prose the loop removes the extraction errors an oracle must not inherit. A geometry-supervised embedding loop also tunes the encoder to the document's vocabulary, so later queries bind colloquial wording to the right entity --- the ingestion improvement. (Mechanics live in the GMS substrate.)

## The full production store
The book's policy store is the same build at scale. It covers every policy, including the prose ones (filing windows, closure notice, escalation windows) that a fee-table-only ingest would miss.

In [ ]:
import torch
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore

def _find_store(name="gms_policy_store_geode"):
    for p in [Path.cwd(), *Path.cwd().parents]:
        c = p / "beyond-prompt-and-pray" / "code" / "data" / name
        if c.exists():
            return c
    raise FileNotFoundError(name + " not found; run scripts/build_geode_rag_store.py")

STORE = _find_store()
store = GMSExpertStore(DocGMSConfig(store_path=str(STORE), ingest_mode="regex"),
                       device=torch.device("cpu"))
assert store.load(), "store failed to load"
print("loaded store:", len(store.adapter.relation_to_idx), "relations,",
      len(store.adapter.entity_to_idx), "entities")

In [ ]:
for r in ['has_fee_amount', 'has_filing_window_days',
          'has_bank_initiated_notice_days', 'has_escalation_window_days']:
    print(f'{r}:', store.query_triples(relation=r)[:1])

## Self-check

In [ ]:
assert res.converged and res.n_triples > 0          # we built a real store
assert res.store.query_triples(head='overdraft')    # and can query it
assert store.query_triples(relation='has_filing_window_days')  # prose policy captured
print('OK - GEODE: building the oracle')